# Transformer track: Hinglish sentiment + zero-shot spam transfer

Runs on a free Colab GPU (Runtime > Change runtime type > T4). About 25 minutes in total.

It uses the **same data, cleaning and test split** as the classical models in the repo, so the numbers are directly comparable:

| Reference (classical, same test set) | Macro-F1 |
|---|---|
| Majority class | 0.223 |
| Char + word TF-IDF, linear SVM | 0.694 (95% CI 0.658 to 0.729) |

Part 1 fine-tunes several encoders on the IIIT-H Hinglish sentiment corpus (3 seeds each). Part 2 trains a spam model on English SMS only and tests it, zero-shot, on Hinglish scam lines.
When it finishes, download `transformer_results.csv` and put it in the repo's `results/` folder.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import re, random, json, numpy as np, pandas as pd, torch, urllib.request, io
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
SEEDS = [42, 43, 44]
RAW = "https://raw.githubusercontent.com/"
URL = RAW + "DrImpossible/Sub-word-LSTM/master/Data/IIITH_Codemixed.txt"
txt = urllib.request.urlopen(URL).read().decode("utf-8", "replace")
_URL, _USER = re.compile(r"https?://\S+|www\.\S+"), re.compile(r"@\w+")
_EMOT = re.compile(r"(?::|;|=)(?:-)?(?:\)|\(|D|P|p)+|:'\(|<3")
clean = lambda t: re.sub(r"\s+", " ", _EMOT.sub(" ", _USER.sub(" ", _URL.sub(" ", t)))).strip()
LAB = {"0": "negative", "1": "neutral", "2": "positive"}
rows = []
for line in txt.splitlines():
    p = line.split("\t")
    if len(p) == 4 and p[2] == p[3] and p[2] in LAB:
        rows.append((clean(p[1]), LAB[p[2]]))
df = pd.DataFrame(rows, columns=["text", "label"]); df = df[df.text.str.len() > 0]
df["key"] = df.text.map(lambda t: re.sub(r"[^a-z0-9\u0900-\u097f ]", "", t.lower()))
df = df.drop_duplicates("key").drop(columns="key").reset_index(drop=True)
tr, te = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)   # identical to src/data2.py
tr, te = tr.reset_index(drop=True), te.reset_index(drop=True)
print(len(tr), len(te), te.label.value_counts().to_dict())   # expect 3100 776

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
LABELS = ["negative", "neutral", "positive"]; l2i = {l: i for i, l in enumerate(LABELS)}
MODELS = ["xlm-roberta-base", "bert-base-multilingual-cased", "google/muril-base-cased", "l3cube-pune/hing-bert", "l3cube-pune/hing-roberta"]

def run(model_name, seed, epochs=4, lr=2e-5, bs=32, max_len=64):
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = lambda d: Dataset.from_dict({"text": d.text.tolist(), "label": [l2i[x] for x in d.label]}).map(
        lambda b: tok(b["text"], truncation=True, max_length=max_len, padding="max_length"), batched=True)
    # validation = 10% of train, used only to pick the best epoch (never the test set)
    t_, v_ = train_test_split(tr, test_size=0.1, stratify=tr.label, random_state=seed)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    args = TrainingArguments(output_dir=f"out/{seed}", num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=bs,
                             per_device_eval_batch_size=64, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
                             metric_for_best_model="f1", seed=seed, report_to=[], fp16=torch.cuda.is_available(), weight_decay=0.01, warmup_ratio=0.1)
    f1m = lambda p: {"f1": f1_score(p.label_ids, p.predictions.argmax(1), average="macro")}
    t = Trainer(model=model, args=args, train_dataset=enc(t_), eval_dataset=enc(v_), compute_metrics=f1m)
    t.train()
    pred = t.predict(enc(te)).predictions.argmax(1); y = [l2i[x] for x in te.label]
    return f1_score(y, pred, average="macro"), accuracy_score(y, pred)

In [ ]:
results = []
for m in MODELS:
    try:
        for s in SEEDS:
            f1, acc = run(m, s); results.append({"model": m, "seed": s, "test_macro_F1": f1, "test_accuracy": acc}); print(m, s, round(f1, 3))
    except Exception as e:
        print("skipped", m, e)
R = pd.DataFrame(results)
S = R.groupby("model").agg(macro_F1_mean=("test_macro_F1", "mean"), macro_F1_std=("test_macro_F1", "std"), acc_mean=("test_accuracy", "mean")).round(4)
S.to_csv("transformer_results.csv"); R.to_csv("transformer_results_by_seed.csv", index=False); S

## Part 2: zero-shot spam transfer (English SMS -> Hinglish)

Does a multilingual encoder trained only on English spam recognise Hinglish scams? The classical English model catches 8% of them at its default threshold.
The Hinglish scam lines come from github.com/gupta7050/multilingual-phishing-detection (about 100 unique lines repeated; we de-duplicate). Read results as indicative only.

In [ ]:
sms = pd.read_csv(io.StringIO(urllib.request.urlopen(RAW + "justmarkham/pycon-2016-tutorial/master/data/sms.tsv").read().decode()), sep="\t", header=None, names=["label", "text"]).drop_duplicates("text")
ch = pd.read_csv(io.StringIO(urllib.request.urlopen(RAW + "gupta7050/multilingual-phishing-detection/main/hinglish_data.csv").read().decode("utf-8-sig"))).drop_duplicates("text")
s_tr, s_te = train_test_split(sms, test_size=0.2, stratify=sms.label, random_state=42)
from sklearn.metrics import roc_auc_score, recall_score
def spam_run(model_name="xlm-roberta-base", seed=42):
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = lambda d: Dataset.from_dict({"text": d.text.tolist(), "label": (d.label == "spam").astype(int).tolist()}).map(lambda b: tok(b["text"], truncation=True, max_length=96, padding="max_length"), batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    args = TrainingArguments(output_dir="out/spam", num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=32, seed=seed, report_to=[], fp16=torch.cuda.is_available(), save_strategy="no")
    t = Trainer(model=model, args=args, train_dataset=enc(s_tr)); t.train()
    out = {}
    for name, d in (("English SMS test", s_te), ("Hinglish challenge (zero-shot)", ch)):
        lg = t.predict(enc(d)).predictions; p = torch.softmax(torch.tensor(lg), 1)[:, 1].numpy(); y = (d.label == "spam").astype(int).values
        out[name] = {"AUC": roc_auc_score(y, p), "spam_recall@0.5": recall_score(y, (p > .5).astype(int)), "false_alarm@0.5": float(((p > .5) & (y == 0)).sum() / (y == 0).sum())}
    return out
spam_res = spam_run(); pd.DataFrame(spam_res).T.round(3).to_csv("transformer_spam_transfer.csv"); pd.DataFrame(spam_res).T.round(3)